# Publications markdown generator for academicpages

Takes a TSV of publications with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `publications.py`. Run either from the `markdown_generator` folder after replacing `publications.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases of citations, rather than Stuart's non-standard TSV format and citation style.


## Data format

The TSV needs to have the following columns: pub_date, title, venue, excerpt, citation, site_url, and paper_url, with a header at the top. 

- `excerpt` and `paper_url` can be blank, but the others must have values. 
- `pub_date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/publications/YYYY-MM-DD-[url_slug]`

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [1]:
!cat publications.tsv

pub_date	title	venue	excerpt	citation	url_slug	paper_url	slides_url	category
2025-01-01	AI-driven materials design: a mini-review	Nature Materials	Comprehensive review of AI-driven approaches for materials design, covering recent advances and future directions in computational materials science.	"Cheng, M., Fu, C., Okabe, R., Chotrattanapituk, A., Boonkird, A., Hung, N.T., & Li, M. (2025). AI-driven materials design: a mini-review. <i>Nature Materials</i>."	ai-driven-materials-design-review		manuscripts
2024-07-05	Structural Constraint Integration in Generative Model for Discovery of Quantum Material Candidates	Nature Materials	We present a novel approach to integrating structural constraints in generative models for quantum material discovery, enabling more accurate predictions of material properties through advanced machine learning techniques.	"Okabe, R., Cheng, M., Chotrattanapituk, A., Mandal, M., Mak, K., Cordova Carrizales, D., Hung, N.T., Fu, X., Han, B., Wang, Y., Xie, W., Cav

## Import pandas

We are using the very handy pandas library for dataframes.

In [2]:
import pandas as pd

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [3]:
publications = pd.read_csv("publications.tsv", sep="\t", header=0)
publications


,pub_date,title,venue,excerpt,citation,url_slug,paper_url,slides_url,category
0,2025-01-01,AI-driven materials design: a mini-review,Nature Materials,Comprehensive review of AI-driven approaches f...,"Cheng, M., Fu, C., Okabe, R., Chotrattanapituk...",ai-driven-materials-design-review,NaN,manuscripts,NaN
1,2024-07-05,Structural Constraint Integration in Generativ...,Nature Materials,We present a novel approach to integrating str...,"Okabe, R., Cheng, M., Chotrattanapituk, A., Ma...",structural-constraint-integration-generative-m...,https://arxiv.org/abs/2407.04557,https://arxiv.org/pdf/2407.04557,manuscripts
2,2024-01-01,Design of efficient solvent-suppression scheme...,Journal of Magnetic Resonance,Development of efficient solvent suppression t...,"Matsunaga, T., Okabe, R., & Ishii, Y. (2024). ...",solvent-suppression-scheme-nmr,NaN,manuscripts,NaN
3,2025-07-01,Tuning Chiral Anomaly Signature in a Dirac Sem...,arXiv,Experimental study demonstrating control of ch...,"Mandal, M., Rha, E., Chotrattanapituk, A., Cor...",chiral-anomaly-dirac-semimetal,https://arxiv.org/abs/2507.17972,manuscripts,NaN
4,2025-01-01,AI-driven defect engineering in advanced therm...,Advanced Materials,Application of AI-driven approaches for defect...,"Fu, C., Cheng, M., Hung, N.T., Rha, E., Chen, ...",ai-defect-engineering-thermoelectric,NaN,manuscripts,NaN
5,2025-01-01,Closing the superconducting gap with AI,Newton,Machine learning approach to predict and close...,"Cheng, M., Okabe, R., & Li, M. (2025). Closing...",closing-superconducting-gap-ai,NaN,manuscripts,NaN
6,2025-01-01,AI-Powered Exploration of Molecular Vibrations...,Digital Discovery,Comprehensive exploration of molecular vibrati...,"Han, B., Okabe, R., Chotrattanapituk, A., Chen...",ai-molecular-vibrations-phonons,NaN,manuscripts,NaN
7,2024-10-01,Large Language Model-Guided Prediction Toward ...,arXiv,Novel application of large language models for...,"Okabe, R., West, Z., Chotrattanapituk, A., Che...",llm-quantum-materials-synthesis,https://arxiv.org/abs/2410.20976,manuscripts,NaN
8,2024-12-01,Quantum Theory of X-ray Photon Correlation Spe...,arXiv,Theoretical framework for X-ray photon correla...,"Siriviboon, P., Fu, C., Landry, M., Okabe, R.,...",https://arxiv.org/abs/2412.03635,manuscripts,NaN,NaN
9,2024-01-01,Ensemble-Embedding Graph Neural Network for Di...,Advanced Materials,Development of ensemble-embedding graph neural...,"Hung, N.T., Okabe, R., Chotrattanapituk, A., &...",NaN,manuscripts,NaN,NaN


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [4]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [6]:
import os
for row, item in publications.iterrows():
    
    md_filename = str(item.pub_date) + "-" + str(item.url_slug) + ".md"
    html_filename = str(item.pub_date) + "-" + str(item.url_slug)
    year = item.pub_date[:4]
    
    ## YAML variables
    
    md = "---\ntitle: \""   + item.title + '"\n'
    
    md += """collection: publications"""
    
    md += """\npermalink: /publication/""" + html_filename
    
    if len(str(item.excerpt)) > 5:
        md += "\nexcerpt: '" + html_escape(item.excerpt) + "'"
    
    md += "\ndate: " + str(item.pub_date) 
    
    md += "\nvenue: '" + html_escape(item.venue) + "'"
    
    if len(str(item.slides_url)) > 5:
        md += "\nslidesurl: '" + item.slides_url + "'"

    if len(str(item.paper_url)) > 5:
        md += "\npaperurl: '" + item.paper_url + "'"
    
    md += "\ncitation: '" + html_escape(item.citation) + "'"
    
    md += "\n---"
    
    ## Markdown description for individual page
        
    if len(str(item.excerpt)) > 5:
        md += "\n" + html_escape(item.excerpt) + "\n"

    if len(str(item.slides_url)) > 5:
        md += "\n[Download slides here](" + item.slides_url + ")\n" 

    if len(str(item.paper_url)) > 5:
        md += "\n[Download paper here](" + item.paper_url + ")\n" 
        
    md += "\nRecommended citation: " + item.citation
    
    md_filename = os.path.basename(md_filename)
       
    with open("../_publications/" + md_filename, 'w') as f:
        f.write(md)

These files are in the publications directory, one directory below where we're working from.

In [ ]:
!ls ../_publications/

2009-10-01-paper-title-number-1.md 2024-02-17-paper-title-number-4.md
2010-10-01-paper-title-number-2.md 2024-07-05-paper-title-number-4.md
2015-10-01-paper-title-number-3.md


In [ ]:
!cat ../_publications/2009-10-01-paper-title-number-1.md

---
title: "Paper Title Number 1"
collection: publications
permalink: /publication/2009-10-01-paper-title-number-1
excerpt: 'This paper is about the number 1. The number 2 is left for future work.'
date: 2009-10-01
venue: 'Journal 1'
slidesurl: 'http://academicpages.github.io/files/slides1.pdf'
paperurl: 'http://academicpages.github.io/files/paper1.pdf'
citation: 'Your Name, You. (2009). &quot;Paper Title Number 1.&quot; <i>Journal 1</i>. 1(1).'
---
This paper is about the number 1. The number 2 is left for future work.

[Download slides here](http://academicpages.github.io/files/slides1.pdf)

[Download paper here](http://academicpages.github.io/files/paper1.pdf)

Recommended citation: Your Name, You. (2009). "Paper Title Number 1." <i>Journal 1</i>. 1(1).